# Adresses email — MSP, CPTS, DAC et DALIA d'Occitanie

Extraction ciblée en vue d'une publication. Les structures concernées souhaitent
rendre leur adresse publique : avant diffusion, on leur applique le même
contrôle qualité que celui du chantier email, pour ne pas publier une adresse
inutilisable ou nominative.

| Catégorie | Libellé |
|---|---|
| 603 | Maison de santé (MSP) |
| 604 | Communauté professionnelle territoriale de santé (CPTS) |
| 606 | Dispositif d'appui à la coordination (DAC) |
| 633 | Structure expérimentale en santé (DALIA) |

Périmètre géographique : Occitanie, soit les treize départements de la région.

**Différence avec les autres notebooks du chantier** : établissements et
personnes morales sont réunis dans une seule table, avec une colonne
`type_structure` pour les distinguer. La logique d'analyse, elle, est identique.

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.chargement import (
    resoudre_tables, charger_telecoms, charger_ege, charger_pm,
)
from src.email_checker import (
    analyser_emails, marquer_doublons, evaluer_correspondance,
    charger_tlds_iana, charger_bases_noms, charger_geo_insee, detecter_geo,
)
from src.autorites import _DEPTS_PAR_REGION, LIBELLES_REGION

pd.set_option("display.max_colwidth", 80)
print("Imports OK")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Périmètre

In [2]:
REGION = "76"
DEPARTEMENTS = set(_DEPTS_PAR_REGION[REGION])

CATEGORIES = {
    "603": "MSP — Maison de santé",
    "604": "CPTS — Communauté professionnelle territoriale de santé",
    "606": "DAC — Dispositif d'appui à la coordination",
    "633": "DALIA — Structure expérimentale en santé",
}

# Une PM est retenue si elle porte elle-même l'une des catégories, ou si elle est
# la personne morale d'un établissement retenu. Passer à False pour ne garder que
# les PM portant directement la catégorie.
INCLURE_PM_PARENTES = True

print(f"Région     : {LIBELLES_REGION[REGION]} ({len(DEPARTEMENTS)} départements)")
print(f"             {sorted(DEPARTEMENTS)}")
print(f"Catégories : {', '.join(CATEGORIES)}")

Région     : Occitanie (13 départements)
             ['09', '11', '12', '30', '31', '32', '34', '46', '48', '65', '66', '81', '82']
Catégories : 603, 604, 606, 633


## 3. Référentiels externes

In [3]:
DOSSIER_REF = Path("../../data/referentiels")

charger_tlds_iana(cache=DOSSIER_REF / "tlds-alpha-by-domain.txt")

df_prenoms    = pd.read_csv(DOSSIER_REF / "prenom.csv",     sep=",", dtype=str)
df_patronymes = pd.read_csv(DOSSIER_REF / "patronymes.csv", sep=",", dtype=str)
charger_bases_noms(df_prenoms, df_patronymes)

df_communes = pd.read_csv(DOSSIER_REF / "communes-france-2026.csv", dtype=str)
stats_geo = charger_geo_insee(df_communes, seuil_population=2000)

from src.email_checker import PRENOMS, PATRONYMES, TLDS_VALIDES
print(f"TLD valides     : {len(TLDS_VALIDES):,}")
print(f"Prénoms retenus : {len(PRENOMS):,}")
print(f"Villes retenues : {stats_geo['villes']:,}")

TLD valides     : 1,437
Prénoms retenus : 1,309
Villes retenues : 5,200


## 4. Chargement et filtrage

In [4]:
TABLES = resoudre_tables(conn)

print("\nTelecoms :")
telecoms = charger_telecoms(conn, TABLES)

print("\nEGE :")
df_ege = charger_ege(conn, TABLES, telecoms)

print("\nPM :")
df_pm = charger_pm(conn, TABLES, telecoms)

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   adresse      -> T_FAC_Adresse
   telecom      -> T_FAC_Telecom
   pm_adresse   -> T_FAC_PmSmsseAdresse
   ege_adresse  -> T_FAC_EgeAdresse
   pm_contact   -> T_FAC_PmSmsseContact
   ege_contact  -> T_FAC_EgeContact

Telecoms :
   3 lignes de canal sentinelle ecartees
   361 doublons (contact, canal) : premiere valeur conservee
   telecoms pivotes : 355,919 contacts, colonnes ['TEL_MAIL', 'TEL_TELECOPIE', 'TEL_TELEPHONE']

EGE :
   228,773 lignes brutes
   228,773 lignes -> 104,650 structures, 150 contacts au maximum pour une structure

PM :
   84,011 lignes brutes
   84,011 lignes -> 56,187 structures, 3 contacts au maximum pour une structure


In [5]:
COL_CAT = "TX_CategorieEntiteGeographiqueExercice"

def normaliser_categorie(serie):
    return serie.fillna("").astype(str).str.strip().str.zfill(3)

# --- Établissements : catégorie et département ---
cat_ege = normaliser_categorie(df_ege[COL_CAT])
sel_ege = cat_ege.isin(CATEGORIES) & df_ege["departement"].isin(DEPARTEMENTS)

ege = df_ege[sel_ege].copy()
ege["code_categorie"] = cat_ege[sel_ege]
ege["origine_selection"] = "CATEGORIE"

print(f"EGE retenus : {len(ege):,}")
print(ege["code_categorie"].map(CATEGORIES).value_counts().to_string())

EGE retenus : 499
code_categorie
MSP — Maison de santé                                      390
CPTS — Communauté professionnelle territoriale de santé     85
DAC — Dispositif d'appui à la coordination                  13
DALIA — Structure expérimentale en santé                    11


In [6]:
# --- Personnes morales ---
cat_pm = normaliser_categorie(df_pm[COL_CAT])
directes = cat_pm.isin(CATEGORIES) & df_pm["departement"].isin(DEPARTEMENTS)

pm = df_pm[directes].copy()
pm["code_categorie"] = cat_pm[directes]
pm["origine_selection"] = "CATEGORIE"
print(f"PM portant directement la categorie : {len(pm):,}")

if INCLURE_PM_PARENTES:
    # categorie de l'etablissement rattache, pour les PM qui n'en portent pas
    cat_par_pm = (ege.dropna(subset=["NB_PmSmsseId"])
                     .assign(cle=lambda d: d["NB_PmSmsseId"].astype(str))
                     .drop_duplicates("cle")
                     .set_index("cle")["code_categorie"])

    deja = set(pm["NB_PmSmsseId"].astype(str))
    a_ajouter = set(cat_par_pm.index) - deja

    parentes = df_pm[df_pm["NB_PmSmsseId"].astype(str).isin(a_ajouter)].copy()
    if len(parentes):
        parentes["code_categorie"] = parentes["NB_PmSmsseId"].astype(str).map(cat_par_pm)
        parentes["origine_selection"] = "PM_PARENTE"
        pm = pd.concat([pm, parentes], ignore_index=True)

    print(f"PM ajoutees comme parentes d'un EGE retenu : {len(parentes):,}")
    manquantes = len(a_ajouter) - len(parentes)
    if manquantes:
        print(f"   {manquantes:,} PM parentes introuvables (inactives ou hors chargement)")

print(f"PM retenues au total : {len(pm):,}")


PM portant directement la categorie : 0
PM ajoutees comme parentes d'un EGE retenu : 498
PM retenues au total : 498


## 5. Table unique

Les deux niveaux sont réunis. `type_structure` distingue les lignes, et les
colonnes propres à chaque niveau sont harmonisées sous un nom commun.

In [7]:
def harmoniser(df, type_structure, col_id, col_finess, col_nom):
    sortie = df.copy()
    sortie["type_structure"] = type_structure
    sortie["identifiant"] = sortie[col_id].astype(str)
    sortie["numero_finess"] = sortie[col_finess]
    sortie["denomination"] = sortie[col_nom]
    sortie["libelle_categorie"] = sortie["code_categorie"].map(CATEGORIES)
    return sortie

structures = pd.concat([
    harmoniser(ege, "EGE", "NB_EgeId", "TX_NumFinessEge", "TX_NomEgeLong"),
    harmoniser(pm, "PM", "NB_PmSmsseId", "TX_NumFinessPm", "TX_DenominationPm"),
], ignore_index=True, sort=False)

print(f"Structures au total : {len(structures):,}")
print()
print(pd.crosstab(structures["libelle_categorie"], structures["type_structure"],
                  margins=True, margins_name="TOTAL").to_string())

Structures au total : 997

type_structure                                           EGE   PM  TOTAL
libelle_categorie                                                       
CPTS — Communauté professionnelle territoriale de santé   85   85    170
DAC — Dispositif d'appui à la coordination                13   13     26
DALIA — Structure expérimentale en santé                  11   10     21
MSP — Maison de santé                                    390  390    780
TOTAL                                                    499  498    997


In [8]:
avec_email = structures["TEL_MAIL"].notna().sum()

print("─" * 56)
print("PANORAMA DES EMAILS — MSP / CPTS / DAC / DALIA OCCITANIE")
print("─" * 56)
print(f"  Structures                  : {len(structures):>8,}")
print(f"  Email renseigné             : {avec_email:>8,}  ({avec_email/len(structures)*100:.1f} %)")
print(f"  Email absent                : {len(structures) - avec_email:>8,}")
print(f"  Emails uniques              : {structures['TEL_MAIL'].nunique():>8,}")
print("─" * 56)
print()
print("Répartition par département :")
print(structures.groupby("departement")["identifiant"].count()
      .rename("structures").to_string())

────────────────────────────────────────────────────────
PANORAMA DES EMAILS — MSP / CPTS / DAC / DALIA OCCITANIE
────────────────────────────────────────────────────────
  Structures                  :      997
  Email renseigné             :      909  (91.2 %)
  Email absent                :       88
  Emails uniques              :      504
────────────────────────────────────────────────────────

Répartition par département :
departement
09     50
11     72
12     80
30    138
31    187
32     26
34    138
46     44
48     26
65     50
66     66
81     70
82     50


## 6. Détection des anomalies

In [9]:
df = analyser_emails(structures, col_email="TEL_MAIL")
df = marquer_doublons(df, col_email_norm="email_norm", col_id="identifiant")
print(f"Analyse terminée — {len(df):,} structures traitées.")

Analyse terminée — 997 structures traitées.


In [10]:
def _corresp_ligne(row):
    return evaluer_correspondance(
        local_part=row["local_part"],
        raison_principale=row.get("denomination", ""),
        raison_parent="",
        adresse=row.get("TX_LibelleVoie", ""),
        libelle_principal="(structure)",
        libelle_parent="",
    )

df["correspondance"] = ""
masque = df["local_part"] != ""
df.loc[masque, "correspondance"] = df.loc[masque].apply(_corresp_ligne, axis=1)

geo = df.apply(lambda row: detecter_geo(row["local_part"], row.get("TX_CogCommune")), axis=1)
df["geo_local"]      = geo.apply(lambda x: x["geo_local"])
df["geo_concordant"] = geo.apply(lambda x: x["geo_concordant"])

print("Correspondances :")
print(df[df["correspondance"] != ""]["correspondance"].value_counts().to_string())

Correspondances :
correspondance
Raison sociale (structure)              565
Aucune correspondance                   305
Raison sociale (structure) + Adresse     38
Adresse                                   1


## 7. Résultats

In [11]:
labels = {0: "Valides", 1: "Critiques (niveau 1)", 2: "Qualité (niveau 2)"}

print("─" * 56)
print("DISTRIBUTION DES SIGNALEMENTS")
print("─" * 56)
for niveau, n in df["niveau_anomalie"].value_counts().sort_index().items():
    print(f"  {labels.get(niveau, niveau):<28} : {n:>6,}  ({n/len(df)*100:5.1f} %)")
print("─" * 56)
print()
print("Par catégorie :")
print(pd.crosstab(df["libelle_categorie"], df["niveau_anomalie"]).to_string())

────────────────────────────────────────────────────────
DISTRIBUTION DES SIGNALEMENTS
────────────────────────────────────────────────────────
  Valides                      :     15  (  1.5 %)
  Critiques (niveau 1)         :     90  (  9.0 %)
  Qualité (niveau 2)           :    892  ( 89.5 %)
────────────────────────────────────────────────────────

Par catégorie :
niveau_anomalie                                          0   1    2
libelle_categorie                                                  
CPTS — Communauté professionnelle territoriale de santé  7   5  158
DAC — Dispositif d'appui à la coordination               3   1   22
DALIA — Structure expérimentale en santé                 0  11   10
MSP — Maison de santé                                    5  73  702


In [12]:
print("Classification de la partie locale :")
classif = df[df["type_local"] != ""]
print(classif["type_local"].value_counts().to_string())
print()
print("Croisement type x catégorie :")
print(pd.crosstab(classif["libelle_categorie"], classif["type_local"]).to_string())

Classification de la partie locale :
type_local
INDETERMINE       571
GENERIQUE         189
INSTITUTIONNEL    146
NOMINATIF           1

Croisement type x catégorie :
type_local                                               GENERIQUE  INDETERMINE  INSTITUTIONNEL  NOMINATIF
libelle_categorie                                                                                         
CPTS — Communauté professionnelle territoriale de santé         47          106              11          1
DAC — Dispositif d'appui à la coordination                      23            2               0          0
DALIA — Structure expérimentale en santé                         4            6               0          0
MSP — Maison de santé                                          115          457             135          0


### Adresses à ne pas publier en l'état

Deux cas à écarter avant diffusion : les adresses inutilisables, et les adresses
nominatives — une adresse au nom d'une personne devient obsolète dès son départ,
et sa publication soulève une question de données personnelles.

In [13]:
NOMINATIFS = ["NOMINATIF", "NOMINATIF_PARTIEL"]

a_ecarter = df[(df["niveau_anomalie"] == 1) | df["type_local"].isin(NOMINATIFS)]
publiables = df[(df["niveau_anomalie"] != 1)
                & (~df["type_local"].isin(NOMINATIFS))
                & (df["code_anomalie"] != "EMAIL_VIDE")]

print(f"Structures analysées      : {len(df):,}")
print(f"  publiables en l'état    : {len(publiables):,}")
print(f"  à écarter ou corriger   : {len(a_ecarter):,}")
print(f"  sans email              : {int((df['code_anomalie'] == 'EMAIL_VIDE').sum()):,}")
print()
print("Motifs d'écartement :")
print(a_ecarter["code_anomalie"].value_counts().to_string())
print()
print(a_ecarter[a_ecarter["type_local"].isin(NOMINATIFS)]["type_local"]
      .value_counts().rename("nominatifs").to_string())

Structures analysées      : 997
  publiables en l'état    : 906
  à écarter ou corriger   : 91
  sans email              : 88

Motifs d'écartement :
code_anomalie
EMAIL_VIDE            88
DOMAINE_SANS_POINT     2
EMAIL_GRAND_PUBLIC     1

type_local
NOMINATIF    1


## 8. Export

In [14]:
from openpyxl.styles import PatternFill, Font, Alignment

DOSSIER_SORTIE = Path("../../results/email")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER = DOSSIER_SORTIE / "annuaire_occitanie_msp_cpts_dac_dalia.xlsx"

COLS = [
    "type_structure", "code_categorie", "libelle_categorie", "origine_selection",
    "numero_finess", "identifiant", "denomination",
    "departement", "TX_CogCommune", "TX_Localite", "adresse_complete",
    "TEL_MAIL", "email_norm", "TEL_TELEPHONE",
    "code_anomalie", "libelle_anomalie", "suggestion",
    "type_local", "type_domaine", "correspondance",
    "geo_local", "geo_concordant",
]
cols = [c for c in COLS if c in df.columns]

def style_entete(ws, couleur):
    for cell in ws[1]:
        cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
        cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 28

def remplir(ws, couleur):
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
            cell.font = Font(name="Arial", size=9)

def auto_width(ws, max_w=45):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

synthese = pd.DataFrame([
    {"Indicateur": "Structures dans le périmètre", "Nombre": len(df)},
    {"Indicateur": "  dont établissements", "Nombre": int((df["type_structure"] == "EGE").sum())},
    {"Indicateur": "  dont personnes morales", "Nombre": int((df["type_structure"] == "PM").sum())},
    {"Indicateur": "—", "Nombre": ""},
    {"Indicateur": "Publiables en l'état", "Nombre": len(publiables)},
    {"Indicateur": "À écarter ou corriger", "Nombre": len(a_ecarter)},
    {"Indicateur": "Sans email", "Nombre": int((df["code_anomalie"] == "EMAIL_VIDE").sum())},
])

FEUILLES = [
    ("Publiables", publiables, "2E7D32", "EBF5EB"),
    ("A_ecarter", a_ecarter, "C62828", "F8CECC"),
    ("Sans_email", df[df["code_anomalie"] == "EMAIL_VIDE"], "757575", "EEEEEE"),
    ("Toutes_structures", df, "1F3864", "F2F6FC"),
]

with pd.ExcelWriter(FICHIER, engine="openpyxl") as writer:
    synthese.to_excel(writer, sheet_name="Synthèse", index=False)
    style_entete(writer.sheets["Synthèse"], "1F3864")
    auto_width(writer.sheets["Synthèse"], max_w=36)

    repartition = pd.crosstab(df["libelle_categorie"], df["type_structure"],
                              margins=True, margins_name="TOTAL").reset_index()
    repartition.to_excel(writer, sheet_name="Par_categorie", index=False)
    style_entete(writer.sheets["Par_categorie"], "1F3864")
    auto_width(writer.sheets["Par_categorie"])

    for nom, donnees, entete, fond in FEUILLES:
        if len(donnees) == 0:
            continue
        donnees[cols].to_excel(writer, sheet_name=nom, index=False)
        style_entete(writer.sheets[nom], entete)
        remplir(writer.sheets[nom], fond)
        auto_width(writer.sheets[nom])

print(f"Export OK → {FICHIER.resolve()}")
print()
for nom, donnees, _, _ in FEUILLES:
    print(f"  {nom:<20} : {len(donnees):,}")

Export OK → /home/jovyan/work/finess+_refonte/signalement_data_contact_finess+/results/email/annuaire_occitanie_msp_cpts_dac_dalia.xlsx

  Publiables           : 906
  A_ecarter            : 91
  Sans_email           : 88
  Toutes_structures    : 997


## Lecture

`Publiables` regroupe les structures dont l'adresse est exploitable et non
nominative : c'est la base de l'annuaire.

`A_ecarter` regroupe les adresses inutilisables et les adresses nominatives. Les
premières demandent une correction, les secondes une adresse de service à la
place — une adresse au nom d'une personne devient obsolète dès son départ et sa
publication pose une question de données personnelles.

`Sans_email` liste les structures qui n'ont aucune adresse renseignée dans
FINESS : ce sont celles à solliciter en priorité si l'annuaire doit être complet.

La colonne `origine_selection` indique comment la structure est entrée dans le
périmètre : `CATEGORIE` si elle porte elle-même l'une des quatre catégories,
`PM_PARENTE` si elle est la personne morale d'un établissement retenu.